In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

A -- [0.44999999999999996, 0, 0.43, 0.41, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.46, 0.87, 0.71, 1.0]
C -- [0.6, 0, 0.35857142857142854, 0.49, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0.93, 1.37, 0.84, 0.97]
D -- [0.65, -1, 0.21285714285714286, -0.55, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.96, 0.54, 1.31, 0.87]
E -- [0.75, -1, 0.23, -0.31, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.46, 0.65, 0.85, 0.89]
F -- [0.8500000000000001, 0, 0.39142857142857146, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1.04, 1.44, 0.7, 0.88]
G -- [0.4, 0, 0.42642857142857143, 0.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.42, 0.36, 1.85, 0.96]
H -- [0.8, 0, 0.5421428571428571, 0.08, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.85, 1.16, 1.02, 0.89]
I -- [0.65, 0, 0.42714285714285716, 1.0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0.86, 1.85, 0.59, 0.76]
K -- [0.75, 1, 0.6764285714285715, -0.23, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1.12, 0.86, 0.99, 1.24]
L -- [0.65, 0, 0.42714285714285716, 0.97, 0, 0, 0, 0, 0, 0, 0, 

In [2]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [4]:
drug_i_want_to_deliver = "CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C"

In [5]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C
Drug Likeness Rules Passed: 4 / 4


In [1]:
pocket_data = model.generate_pocket_data(temperature = 0.2)
print(len(pocket_data))
print(type(pocket_data))

NameError: name 'model' is not defined

In [7]:
pocket_data

{'SMILES': 'CC(C)OC(=O)Nc1cc2cc(c(F)c(N)c2cn1)-c1cnc2OCCNc2c1C',
 '3D_skeleton': [[8.411924223532136, -12.420471774453306, -1.1855572218776074],
  [13.859666583963813, 0.3004059950517176, -2.928075523382878],
  [10.452725607653917, -7.281986204380749, -5.661232661861408],
  [-2.2453880481258124, -12.170333439733522, -5.530846568909546],
  [-12.5314866449292, -4.173557307446534, -1.7844430567753267],
  [-1.0452583198978846, -12.652864071644965, 2.646032233837534],
  [-9.255100604153952, -6.713238748397894, -5.374905600606836],
  [3.090775031208509, -11.3233622867166, 3.5927064868109815],
  [8.60861718358527, 8.310590847727633, -0.2543880392503147],
  [9.144936666062076, -0.4034619211192937, 7.706731343223038],
  [9.771253787068899, 0.460142289894323, -6.583291547296572],
  [4.071320760410369, -3.285844982705443, -10.407268829807217],
  [-4.207960449181804, -8.529511779665986, 6.316552482382627],
  [4.160906801208518, -10.451718704575521, 2.16367038482045],
  [-0.061630555175244385, 10.6

In [8]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [9]:
# GENERATION QUALITY ASSESSMENTS

In [10]:
import torch
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=10):
    model.eval()
    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(coord_context, bioch_context, coordinate).unsqueeze(0).to(map4_enc.device)
        output, _ = model.forward(naano_X, map4_enc)

        # print raw predicted vector
        print(f"\nPosition {i}:")
        print(f"  raw output: {output[0].detach().cpu().numpy().round(2)}")

        # print distances to all amino acids
        vector = output[0].detach().float()
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = torch.norm(vector - n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        print(f"  top 3: {sorted_distances[:3]}")

        # update context
        aa_id = sorted_distances[0][0]
        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate.tolist() if torch.is_tensor(coordinate) else coordinate]


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=model._pocket_sph_skeleton())


Position 0:
  raw output: [ 0.63  0.2   0.24 -0.15  0.02 -0.36 -0.35 -0.28  0.4  -1.31  0.2  -0.65
 -0.7   0.74  0.78  1.26  0.04 -0.13  0.71  0.94  0.23]
  top 3: [('T', 2.5165717601776123), ('S', 2.609964609146118), ('G', 2.8023717403411865)]

Position 1:
  raw output: [ 0.29 -0.05  0.83  0.37 -0.9  -0.14 -0.24 -0.77 -0.08 -0.91  0.34  0.41
 -0.11  0.42 -0.52  0.89 -0.53 -0.36  0.29  0.66  0.28]
  top 3: [('T', 2.581991195678711), ('G', 2.6323745250701904), ('S', 2.743633508682251)]

Position 2:
  raw output: [ 0.04  0.96  0.9   0.32  0.22 -0.08  0.44 -0.64  0.67 -0.36  0.82 -0.54
 -1.    0.72 -0.25  0.89 -0.57 -0.2   0.75  0.22  0.44]
  top 3: [('T', 2.7958054542541504), ('S', 3.0245890617370605), ('N', 3.0613596439361572)]

Position 3:
  raw output: [-0.11  0.99  0.98  0.34  0.11  0.2   0.37 -0.65  0.51 -0.14  0.89 -0.4
 -1.11  0.76 -0.53  0.58 -0.66 -0.23  0.61 -0.24  0.49]
  top 3: [('T', 3.1116747856140137), ('A', 3.2239322662353516), ('C', 3.2260217666625977)]

Position 4:
  r